# Qualité de l'eau : compléments à `main.ipynb`

Notebook de la partie Eau. Il garde une statistique descriptive des indices biologiques retirée du rapport final : l'évolution 2013-2023 de chaque indice par zone hydrographique (ancienne partie 2.1.4 de `main.ipynb`).

À lancer après `main.ipynb`, qui écrit `data/processed/eau_zones_hydro.parquet`.

In [ ]:
import os
from pathlib import Path

for dossier in [Path.cwd(), *Path.cwd().parents]:  # se placer à la racine du dépôt
    if (dossier / "src").is_dir():
        os.chdir(dossier)
        break

import matplotlib.pyplot as plt
import pandas as pd

from src.geo.recuperer_zones_hydro import recuperer_zones_hydro
from src.visualisation.tracer_carte import tracer_carte

eau_zones_hydro = pd.read_parquet("data/processed/eau_zones_hydro.parquet")
zones_hydro = recuperer_zones_hydro()
zones_hydro = zones_hydro[~zones_hydro["code_zone"].str.startswith("12")]  # zones de Corse

## Évolution 2013-2023 par zone hydrographique

Taux d'évolution de chaque indice entre la première et la dernière année, en %, pour les zones suivies ces deux années-là. Les palettes sont orientées pour que, sur les trois cartes, **bleu = amélioration** et **rouge = dégradation** : pour l'IPR, une hausse est une dégradation.

In [ ]:
ANNEE_DEBUT, ANNEE_FIN = 2013, 2023
debut = eau_zones_hydro[eau_zones_hydro["annee"] == ANNEE_DEBUT].set_index("code_zone")[["I2M2", "IBD", "IPR"]]
fin = eau_zones_hydro[eau_zones_hydro["annee"] == ANNEE_FIN].set_index("code_zone")[["I2M2", "IBD", "IPR"]]
evolution = ((fin - debut) / debut.where(debut > 0) * 100).reset_index()  # pas de taux si la valeur de départ est nulle
print("zones avec un taux :", evolution[["I2M2", "IBD", "IPR"]].count().to_dict())

fig, axes = plt.subplots(1, 3, figsize=(16, 5.5))
tracer_carte(zones_hydro, evolution, "I2M2", f"I2M2, évolution {ANNEE_DEBUT}-{ANNEE_FIN} (%)", axes[0], cmap="RdBu", centrer=True)
tracer_carte(zones_hydro, evolution, "IBD", f"IBD, évolution {ANNEE_DEBUT}-{ANNEE_FIN} (%)", axes[1], cmap="RdBu", centrer=True)
tracer_carte(zones_hydro, evolution, "IPR", f"IPR, évolution {ANNEE_DEBUT}-{ANNEE_FIN} (%)", axes[2], cmap="RdBu_r", centrer=True)

Il n'y a pas de tendance d'ensemble : entre 2013 et 2023, l'I2M2 progresse dans environ la moitié des zones, l'IBD dans 46 % d'entre elles, et l'IPR s'améliore dans 40 % seulement. Aucune région ne se détache nettement, les zones qui s'améliorent et celles qui se dégradent sont souvent voisines.

Ces cartes sont à lire avec prudence :
- seules les zones suivies en 2013 **et** en 2023 ont un taux (environ 1 400 pour l'I2M2, 2 200 pour l'IBD, 1 000 pour l'IPR). Pour l'I2M2, l'ouest de la France manque presque entièrement : le réseau n'a été étendu qu'en 2015 ;
- chaque taux compare deux années isolées, souvent une seule mesure par zone : une partie des écarts est du bruit d'une année sur l'autre.